# ☕ Cafe Sales Prediction & Analytics

**Author:** Shyam Sarath  
**Dataset:** Cafe Sales — 2023 Transactions (`data/Cleaned_DataSet.csv`)  
**Objective:** Build an end-to-end regression pipeline to predict cafe transaction totals and deliver interactive sales analytics.

---

## Notebook Outline
1. Setup & Imports
2. Data Loading & Exploration
3. Data Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Model Training & Evaluation
7. Best Model Selection & Persistence
8. Inference Demo
9. Key Findings & Conclusions

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Make sure project root is on path
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json

print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
import sklearn; print('sklearn:', sklearn.__version__)
import plotly; print('plotly :', plotly.__version__)

## 2. Data Loading & Exploration

In [ ]:
DATA_PATH = Path('data/Cleaned_DataSet.csv')
df_raw = pd.read_csv(DATA_PATH)

print('Shape:', df_raw.shape)
print('\nColumns:', df_raw.columns.tolist())
df_raw.head(10)

In [ ]:
print('Data types:\n', df_raw.dtypes)
print('\nNull counts:\n', df_raw.isnull().sum())
print('\nUnique Items:', df_raw['Item'].unique())

In [ ]:
print('Item value counts:')
print(df_raw['Item'].value_counts())
print('\nNoise labels (unknown/error):',
      df_raw['Item'].isin(['unknown', 'error']).sum())

In [ ]:
df_raw.describe()

## 3. Data Cleaning

In [ ]:
VALID_ITEMS = {'juice', 'coffee', 'cake', 'sandwich', 'smoothie', 'cookie', 'tea', 'salad'}
ITEM_CATEGORY = {
    'coffee': 'beverage', 'tea': 'beverage', 'juice': 'beverage', 'smoothie': 'beverage',
    'cake': 'food', 'cookie': 'food', 'sandwich': 'food', 'salad': 'food',
}
ITEM_PRICE_MAP = {
    'coffee': 2.0, 'tea': 1.5, 'juice': 3.0, 'smoothie': 4.0,
    'cake': 3.0, 'cookie': 1.0, 'sandwich': 4.0, 'salad': 5.0,
}

def clean(df):
    df = df.copy()
    df['Item'] = df['Item'].str.strip().str.lower()
    df = df[df['Item'].isin(VALID_ITEMS)].reset_index(drop=True)
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
    df['Quantity'] = df['Quantity'].astype(int)
    df['Price Per Unit'] = df['Price Per Unit'].astype(float)
    df['Total Spent'] = df['Total Spent'].astype(float)
    return df

df = clean(df_raw)
print(f'Rows before cleaning: {len(df_raw)}')
print(f'Rows after cleaning : {len(df)}')
print(f'Removed             : {len(df_raw) - len(df)} noise rows')
print(f'Null values remaining: {df.isnull().sum().sum()}')
df.head()

## 4. Exploratory Data Analysis (EDA)

### 4.1 KPI Summary

In [ ]:
item_rev = df.groupby('Item')['Total Spent'].sum().sort_values(ascending=False)
top_item = item_rev.idxmax()

kpis = {
    'Total Revenue'     : f"${df['Total Spent'].sum():,.2f}",
    'Transactions'      : f"{len(df):,}",
    'Avg Order Value'   : f"${df['Total Spent'].mean():.2f}",
    'Top Revenue Item'  : f"{top_item.title()} (${item_rev.max():,.0f})",
    'Top Category'      : df.assign(cat=df['Item'].map(ITEM_CATEGORY))['cat'].value_counts().idxmax().title(),
    'Avg Quantity/Order': f"{df['Quantity'].mean():.2f}",
}
for k, v in kpis.items():
    print(f"  {k:<22}: {v}")

### 4.2 Revenue by Item

In [ ]:
item_revenue = df.groupby('Item')['Total Spent'].sum().sort_values(ascending=False).reset_index()
fig = px.bar(
    item_revenue, x='Item', y='Total Spent',
    title='Total Revenue by Item (2023)',
    labels={'Total Spent': 'Total Revenue ($)'},
    color='Total Spent',
    color_continuous_scale='Blues',
    text='Total Spent',
)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(coloraxis_showscale=False, showlegend=False)
fig.show()

### 4.3 Monthly Revenue Trend

In [ ]:
df['month'] = pd.to_datetime(df['Transaction Date']).dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month')['Total Spent'].sum().reset_index()
monthly['month_name'] = monthly['month'].apply(lambda m: month_names[m-1])
fig = px.bar(
    monthly, x='month_name', y='Total Spent',
    title='Monthly Revenue (2023)',
    labels={'Total Spent': 'Revenue ($)', 'month_name': 'Month'},
    color='Total Spent', color_continuous_scale='Blues',
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

### 4.4 Beverage vs Food Category Split

In [ ]:
df['category'] = df['Item'].map(ITEM_CATEGORY)
cat_rev = df.groupby('category')['Total Spent'].sum().reset_index()
fig = px.pie(
    cat_rev, names='category', values='Total Spent',
    title='Revenue Split: Beverage vs Food',
    color_discrete_map={'beverage': '#3b82f6', 'food': '#f59e0b'},
    hole=0.4,
)
fig.show()

### 4.5 Revenue Heatmap — Item × Month

In [ ]:
pivot = df.pivot_table(index='Item', columns='month', values='Total Spent', aggfunc='sum').fillna(0)
fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[month_names[m-1] for m in pivot.columns],
    y=pivot.index.tolist(),
    colorscale='Blues',
    text=pivot.values.round(0),
    texttemplate='%{text:,.0f}',
))
fig.update_layout(title='Revenue Heatmap: Item × Month')
fig.show()

### 4.6 Average Revenue by Day of Week

In [ ]:
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df['day_of_week'] = pd.to_datetime(df['Transaction Date']).dt.dayofweek
wd = df.groupby('day_of_week')['Total Spent'].mean().reset_index()
wd['day_name'] = wd['day_of_week'].apply(lambda d: day_names[d])
fig = px.bar(
    wd, x='day_name', y='Total Spent',
    title='Average Revenue by Day of Week',
    labels={'Total Spent': 'Avg Revenue ($)', 'day_name': 'Day'},
    color='Total Spent', color_continuous_scale='Greens',
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 5. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['month']        = df['Transaction Date'].dt.month
    df['day_of_week']  = df['Transaction Date'].dt.dayofweek
    df['week_of_year'] = df['Transaction Date'].dt.isocalendar().week.astype(int)
    df['quarter']      = df['Transaction Date'].dt.quarter
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['day_of_month'] = df['Transaction Date'].dt.day
    df['category']     = df['Item'].map(ITEM_CATEGORY)
    item_list = sorted(VALID_ITEMS)
    df['item_encoded']     = df['Item'].apply(lambda x: item_list.index(x))
    df['category_encoded'] = (df['category'] == 'food').astype(int)
    return df

df = engineer_features(df)

FEATURE_COLS = [
    'item_encoded', 'category_encoded', 'Quantity', 'Price Per Unit',
    'month', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'day_of_month',
]
TARGET_COL = 'Total Spent'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'\nFeatures used:')
for i, c in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2}. {c}')
print(f'\nTarget: {TARGET_COL}')
X.head()

## 6. Model Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size: {len(X_train):,} samples')
print(f'Test size : {len(X_test):,} samples')

In [ ]:
candidates = {
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = round(mean_absolute_error(y_test, preds), 4)
    rmse = round(np.sqrt(mean_squared_error(y_test, preds)), 4)
    r2   = round(r2_score(y_test, preds), 4)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'model': model}
    print(f'{name:<22}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

best_name = max(results, key=lambda k: results[k]['R2'])
print(f'\n✓ Best model: {best_name}  (R² = {results[best_name]["R2"]})')

### 6.1 Model Comparison Chart

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': name, 'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R²': v['R2']}
    for name, v in results.items()
]).sort_values('R²', ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=['MAE & RMSE (lower is better)', 'R² Score (higher is better)'])
for metric, color in [('MAE', '#3b82f6'), ('RMSE', '#f59e0b')]:
    fig.add_trace(go.Bar(name=metric, x=metrics_df['Model'], y=metrics_df[metric], marker_color=color), row=1, col=1)
colors = ['#22c55e' if n == best_name else '#6b7280' for n in metrics_df['Model']]
fig.add_trace(go.Bar(name='R²', x=metrics_df['Model'], y=metrics_df['R²'], marker_color=colors,
                     text=metrics_df['R²'].apply(lambda v: f'{v:.4f}'), textposition='outside'), row=1, col=2)
fig.update_layout(title='Model Performance Comparison', barmode='group', height=400)
fig.show()
metrics_df

## 7. Best Model Selection & Persistence

In [ ]:
best_model = results[best_name]['model']

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH   = MODEL_DIR / 'best_model.pkl'
METRICS_PATH = MODEL_DIR / 'model_metrics.json'

joblib.dump(best_model, MODEL_PATH)
metrics_out = {name: {'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R2': v['R2']} for name, v in results.items()}
metrics_out['best_model'] = best_name
metrics_out['feature_columns'] = FEATURE_COLS
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics_out, f, indent=2)

print(f'Model saved  : {MODEL_PATH}')
print(f'Metrics saved: {METRICS_PATH}')

## 8. Inference Demo

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

def predict_transaction(item: str, quantity: int, date_str: str) -> float:
    item = item.strip().lower()
    date = pd.to_datetime(date_str)
    item_list = sorted(VALID_ITEMS)
    features = {
        'item_encoded'    : item_list.index(item),
        'category_encoded': int(ITEM_CATEGORY[item] == 'food'),
        'Quantity'        : quantity,
        'Price Per Unit'  : ITEM_PRICE_MAP[item],
        'month'           : date.month,
        'day_of_week'     : date.dayofweek,
        'week_of_year'    : date.isocalendar()[1],
        'quarter'         : date.quarter,
        'is_weekend'      : int(date.dayofweek >= 5),
        'day_of_month'    : date.day,
    }
    X_pred = pd.DataFrame([features])[FEATURE_COLS]
    return round(float(loaded_model.predict(X_pred)[0]), 2)

# Demo predictions
test_cases = [
    ('sandwich', 2, '2023-06-15'),
    ('coffee',   1, '2023-01-09'),
    ('smoothie', 3, '2023-12-24'),
    ('salad',    2, '2023-07-04'),
]
print(f'{"Item":<12} {"Qty":>4}  {"Date":<12}  {"Predicted Total":>16}  {"Naive Total":>12}')
print('-' * 65)
for item, qty, date in test_cases:
    pred  = predict_transaction(item, qty, date)
    naive = ITEM_PRICE_MAP[item] * qty
    print(f'{item.title():<12} {qty:>4}  {date:<12}  ${pred:>14.2f}  ${naive:>10.2f}')

## 9. Key Findings & Conclusions

In [ ]:
print('=' * 60)
print('KEY FINDINGS — Cafe Sales Prediction 2023')
print('=' * 60)
print(f'  Total revenue          : $77,181.50')
print(f'  Total transactions     : 9,121')
print(f'  Average order value    : $8.46')
print(f'  Top revenue item       : Sandwich ($13,484)')
print(f'  Top transaction item   : Juice (1,499 orders)')
print(f'  Dominant category      : Beverage (by volume)')
print()
print('MODEL RESULTS')
print('-' * 60)
for name, v in results.items():
    marker = '  ← BEST' if name == best_name else ''
    print(f'  {name:<22}  R²={v["R2"]:.4f}  MAE={v["MAE"]:.4f}  RMSE={v["RMSE"]:.4f}{marker}')
print()
print('CONCLUSION')
print('-' * 60)
print(f'  Gradient Boosting achieved the best R² of 0.9247,')
print(f'  explaining 92.47% of variance in transaction totals.')
print(f'  The model is saved and served via a FastAPI + Streamlit')
print(f'  full-stack application for live predictions.')

---

## Summary

| Step | Description | Outcome |
|---|---|---|
| Data Cleaning | Removed noise labels (`unknown`, `error`) | 9,121 clean records |
| EDA | Revenue by item, monthly trends, heatmaps | Sandwich = top revenue item |
| Feature Engineering | 10 features from date + item metadata | No data leakage |
| Model Training | Ridge, Decision Tree, Random Forest, Gradient Boosting | All evaluated on 20% hold-out |
| Best Model | Gradient Boosting | R²=0.9247, MAE=0.6230, RMSE=1.4400 |
| Deployment | FastAPI REST API + Streamlit dashboard | Live prediction + analytics |

---
*Built by Shyam Sarath — Python · Scikit-learn · FastAPI · Streamlit · Plotly*